In [ ]:
# ==============================
# Federated Learning on Fashion-MNIST (Dirichlet Non-IID)
# Model: LeNet++ CNN
# Algorithm: FedProx + Masking + Adaptive Noise
# Tracks: Train/Val/Test Loss & Accuracy + Round Time + Total Time
# ==============================

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import numpy as np
import random
import copy
from collections import OrderedDict
import time

# -------------------------
# Configuration
# -------------------------
NUM_CLIENTS = 100
CLIENT_FRAC = 0.3
ROUNDS = 60
LOCAL_EPOCHS = 3
BATCH_SIZE = 128
LR = 0.01
MU = 0.01
MASK_SCALE = 0.001
ADAPTIVE_NOISE_FACTOR = 0.2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
ALPHA = 0.1

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.benchmark = True

print("Device:", DEVICE)

# -------------------------
# Dataset: Fashion-MNIST
# -------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform
)

test_dataset = datasets.FashionMNIST(
    root="./data", train=False, download=True, transform=transform
)

# -------------------------
# Validation Split: 10 fixed clients
# -------------------------
VAL_CLIENTS = list(range(10))
TRAIN_CLIENTS = list(range(10, NUM_CLIENTS))

# -------------------------
# Dirichlet Non-IID Partition
# -------------------------
def dirichlet_partition(dataset, clients, alpha=0.1):
    num_classes = 10
    labels = np.array(dataset.targets)
    idxs = np.arange(len(labels))
    client_dict = {i: [] for i in clients}

    class_indices = [idxs[labels == i] for i in range(num_classes)]

    for c_idx in class_indices:
        np.random.shuffle(c_idx)

        proportions = np.random.dirichlet(alpha * np.ones(len(clients)))
        proportions = (np.cumsum(proportions) * len(c_idx)).astype(int)
        proportions = np.concatenate(([0], proportions))

        for i, client in enumerate(clients):
            client_dict[client] += list(c_idx[proportions[i]:proportions[i+1]])

    for client in clients:
        np.random.shuffle(client_dict[client])

    return client_dict

client_data = dirichlet_partition(
    train_dataset,
    list(range(NUM_CLIENTS)),
    alpha=ALPHA
)

# -------------------------
# Model: CNN (LeNet++)
# -------------------------
class FMNIST_CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)

        self.pool = nn.MaxPool2d(2)

        self.fc1 = nn.Linear(64 * 7 * 7, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))

        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))

        return self.fc2(x)

# -------------------------
# FedProx Regularizer
# -------------------------
def fedprox_loss(model, global_model, mu):
    prox = 0.0

    for w, w0 in zip(model.parameters(), global_model.parameters()):
        prox += torch.sum((w - w0.detach()) ** 2)

    return mu / 2 * prox

# -------------------------
# Model <-> Vector Utils
# -------------------------
def model_to_vector(model):
    return torch.cat([p.data.view(-1) for p in model.parameters()])

def vector_to_model(model, vec):
    pointer = 0

    for p in model.parameters():
        num = p.numel()

        p.data = vec[pointer:pointer + num].view_as(p).clone()

        pointer += num

# -------------------------
# Evaluation
# -------------------------
def evaluate(model, loader):
    model.eval()

    correct, total, loss_val = 0, 0, 0.0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)

            out = model(x)

            loss_val += F.cross_entropy(out, y).item() * y.size(0)

            pred = out.argmax(dim=1)

            correct += (pred == y).sum().item()
            total += y.size(0)

    return 100 * correct / total, loss_val / total

# -------------------------
# Federated Training
# -------------------------
global_model = FMNIST_CNN().to(DEVICE)

global_vec = model_to_vector(global_model)

train_acc, val_acc, test_acc = [], [], []

loss_curve = []
train_loss_curve = []
val_loss_curve = []
test_loss_curve = []

round_time = []
total_time = []

start_total = time.time()

for rnd in range(ROUNDS):

    start_round = time.time()

    selected_clients = random.sample(
        TRAIN_CLIENTS,
        int(NUM_CLIENTS * CLIENT_FRAC)
    )

    # ------------------------
    # Ring-based pairwise masks
    # ------------------------
    masks = {
        cid: torch.zeros_like(global_vec)
        for cid in selected_clients
    }

    sorted_ids = sorted(selected_clients)

    K = len(sorted_ids)

    for idx, cid in enumerate(sorted_ids):

        left = sorted_ids[(idx - 1) % K]
        right = sorted_ids[(idx + 1) % K]

        if cid < right:

            rmask = torch.randn_like(global_vec) * MASK_SCALE

            masks[cid] += rmask
            masks[right] -= rmask

    updates = []
    local_losses = []

    for cid in selected_clients:

        local_model = copy.deepcopy(global_model)

        local_model.train()

        optimizer = optim.SGD(
            local_model.parameters(),
            lr=LR,
            momentum=0.9,
            weight_decay=5e-4
        )

        loader = DataLoader(
            Subset(train_dataset, client_data[cid]),
            batch_size=BATCH_SIZE,
            shuffle=True
        )

        for _ in range(LOCAL_EPOCHS):

            for x, y in loader:

                x, y = x.to(DEVICE), y.to(DEVICE)

                optimizer.zero_grad()

                loss = F.cross_entropy(local_model(x), y)

                loss += fedprox_loss(local_model, global_model, MU)

                loss.backward()

                optimizer.step()

        local_losses.append(loss.item())

        # ------------------------
        # Mask + Adaptive Noise
        # ------------------------
        delta = model_to_vector(local_model) - global_vec

        pointer = 0

        for p in local_model.parameters():

            num = p.numel()

            layer_update = delta[pointer:pointer + num].view_as(p)

            layer_norm = torch.norm(layer_update)

            noise = (
                torch.randn_like(layer_update)
                * layer_norm
                * ADAPTIVE_NOISE_FACTOR
            )

            delta[pointer:pointer + num] += noise.view(-1)

            pointer += num

        # Add mask
        delta += masks[cid]

        updates.append(delta)

    # ------------------------
    # Aggregate Updates
    # ------------------------
    avg_update = torch.stack(updates).mean(0)

    global_vec += avg_update

    vector_to_model(global_model, global_vec)

    # ------------------------
    # Metrics
    # ------------------------
    tr_acc, tr_loss = evaluate(
        global_model,
        DataLoader(
            torch.utils.data.ConcatDataset([
                Subset(train_dataset, client_data[c])
                for c in selected_clients
            ]),
            batch_size=BATCH_SIZE
        )
    )

    va_acc, va_loss = evaluate(
        global_model,
        DataLoader(
            torch.utils.data.ConcatDataset([
                Subset(train_dataset, client_data[c])
                for c in VAL_CLIENTS
            ]),
            batch_size=BATCH_SIZE
        )
    )

    te_acc, te_loss = evaluate(
        global_model,
        DataLoader(test_dataset, batch_size=BATCH_SIZE)
    )

    train_acc.append(tr_acc)
    val_acc.append(va_acc)
    test_acc.append(te_acc)

    loss_curve.append(np.mean(local_losses))

    train_loss_curve.append(tr_loss)
    val_loss_curve.append(va_loss)
    test_loss_curve.append(te_loss)

    rt = time.time() - start_round

    round_time.append(rt)

    total_time.append(time.time() - start_total)

    print(
        f"Round {rnd+1:03d} | "
        f"LocalLoss {loss_curve[-1]:.4f} | "
        f"TrainLoss {tr_loss:.4f} | "
        f"ValLoss {va_loss:.4f} | "
        f"TestLoss {te_loss:.4f} | "
        f"Train {tr_acc:.4f} | "
        f"Val {va_acc:.4f} | "
        f"Test {te_acc:.4f} | "
        f"RoundTime {rt:.2f}s | "
        f"TotalTime {total_time[-1]:.2f}s"
    )

Device: cuda
Round 001 | LocalLoss 0.8727 | TrainLoss 2.1081 | ValLoss 2.3853 | TestLoss 2.3544 | Train 25.0396 | Val 17.5502 | Test 16.5400 | RoundTime 26.78s | TotalTime 26.78s
Round 002 | LocalLoss 0.6409 | TrainLoss 2.1546 | ValLoss 2.1722 | TestLoss 2.3235 | Train 24.8711 | Val 25.2481 | Test 19.8500 | RoundTime 20.60s | TotalTime 47.38s
Round 003 | LocalLoss 0.4399 | TrainLoss 1.7111 | ValLoss 1.9844 | TestLoss 1.8302 | Train 41.0328 | Val 27.3542 | Test 32.1200 | RoundTime 22.93s | TotalTime 70.31s
Round 004 | LocalLoss 0.4533 | TrainLoss 1.6984 | ValLoss 2.1058 | TestLoss 2.0520 | Train 43.5355 | Val 31.5178 | Test 30.5800 | RoundTime 16.00s | TotalTime 86.30s
Round 005 | LocalLoss 0.3140 | TrainLoss 1.6468 | ValLoss 2.0670 | TestLoss 1.8646 | Train 48.9681 | Val 34.8584 | Test 42.0400 | RoundTime 21.26s | TotalTime 107.56s
Round 006 | LocalLoss 0.3149 | TrainLoss 1.5629 | ValLoss 2.0291 | TestLoss 1.8157 | Train 45.4750 | Val 22.9726 | Test 33.7500 | RoundTime 23.13s | TotalTi

In [ ]:
# -------------------------
# Save metrics to CSV
# -------------------------
import csv

metrics = {
    "round": list(range(1, ROUNDS+1)),
    "train_acc": train_acc,
    "val_acc": val_acc,
    "test_acc": test_acc,
    "local_loss": loss_curve,
    "train_loss": train_loss_curve,
    "val_loss": val_loss_curve,
    "test_loss": test_loss_curve,
    "round_time": round_time,
    "total_time": total_time
}

with open("FMNIST_masking_federated_metrics_2.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(metrics.keys())
    writer.writerows(zip(*metrics.values()))

# Colab download
try:
    from google.colab import files
    files.download("FMNIST_masking_federated_metrics_2.csv")
except:
    print("CSV saved locally as FMNIST_masking_federated_metrics_2.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# -------------------------
# Save Global Model
# -------------------------
MODEL_PATH = "FPMAN_fmnist_2.pth"

torch.save(global_model.state_dict(), MODEL_PATH)

print("Model saved to:", MODEL_PATH)

# -------------------------
# Download model to laptop
# -------------------------
from google.colab import files
files.download(MODEL_PATH)

Model saved to: FPMAN_fmnist.pth


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>